# Workshop 4 — Fase 3: información latente

Evalúa la hipótesis de un predictor marginal antes de elegir otra arquitectura.
Ejecuta solamente `src.latent_diagnostic_runner`: baseline de TODO `train.x`,
evaluación del mismo prefijo de 128 secuencias de `val` y, opcionalmente, dos
modelos AE z32 nuevos (`identity` / `tanh`), con 24 updates como máximo cada uno.
No reanuda checkpoints ni llama a la matriz de 12 experimentos.

Los checkpoints externos son de **solo lectura**, AE con vocabulario y contrato
compatibles. Conserva los históricos y ejecuta con una carpeta nueva.
Los pilotos T4 anteriores no guardaron sus pesos: el experimento corto nuevo
no es una evaluación retrospectiva de aquellos modelos.

El ZIP no incluye Shakespeare ni pesos. Se usan los Parquet ya existentes.
Un prefijo fijo y una ejecución corta no demuestran estabilidad por 30 épocas.


In [ ]:
from google.colab import drive, files
drive.mount("/content/drive")
from pathlib import Path
from datetime import datetime, timezone
import json, subprocess, sys, zipfile


In [ ]:
# @title Rutas (ZIP nuevo y carpeta de extracción nueva)
ZIP_SOURCE = "Upload" # @param ["Upload", "Drive"]
PROJECT_ZIP = "/content/drive/MyDrive/Workshop_4/workshop4_colab_latent_phase3.zip" # @param {type:"string"}
EXTRACT_DIR = "/content/workshop4_latent_phase3" # @param {type:"string"}
DRIVE_PROCESSED_DIR = "/content/drive/MyDrive/Workshop_4/data/processed" # @param {type:"string"}
LOCAL_PROCESSED_DIR = "/content/workshop4_latent_data/processed" # @param {type:"string"}
OUTPUT_PARENT = "/content/drive/MyDrive/Workshop_4/latent_phase3" # @param {type:"string"}
# Rutas estáticas opcionales a AE históricos. Vacío = no cargar modelo histórico.
BEST_CHECKPOINT = "" # @param {type:"string"}
LAST_CHECKPOINT = "" # @param {type:"string"}


In [ ]:
if ZIP_SOURCE == "Upload":
    uploaded = files.upload()
    if len(uploaded) != 1:
        raise ValueError("Sube exactamente el ZIP nuevo de fase 3")
    PROJECT_ZIP = next(iter(uploaded))
zip_path = Path(PROJECT_ZIP).expanduser().resolve()
with zipfile.ZipFile(zip_path) as archive:
    helpers = [n for n in archive.namelist() if n.endswith("Workshop_4/src/colab_utils.py")]
    if len(helpers) != 1:
        raise ValueError("ZIP de proyecto incompatible")
    zip_import_root = helpers[0].removesuffix("src/colab_utils.py")
sys.path.insert(0, str(zip_path) + "/" + zip_import_root)
from src.colab_utils import extract_project_zip, stage_processed_parquet, runtime_info
WORKSHOP_DIR = extract_project_zip(zip_path, EXTRACT_DIR)
assert (WORKSHOP_DIR / "src/latent_diagnostic_runner.py").is_file()
# Copia solo train/val, verificando hashes. Nunca descarga, modifica ni carga test.
PROCESSED_DIR = stage_processed_parquet(DRIVE_PROCESSED_DIR, LOCAL_PROCESSED_DIR)
print(runtime_info())
print("Código:", WORKSHOP_DIR, "Datos:", PROCESSED_DIR)


## Panel de ejecución

`RUN_SHORT_PROBE=False` calcula baselines y evalúa los checkpoints indicados,
con cero actualizaciones. `True` añade dos modelos nuevos: pesos iniciales
idénticos, Adam nuevo y el mismo prefijo de 3.072 ejemplos permutado con semilla
42. Snapshots 0, 1, 3, 6, 12 y 24; FP64 para normas/clipping y métricas.
Máximo 48 actualizaciones entre ambos, 180 s por modelo y 600 s por proceso.

Revisar primero el plan. `EXECUTE=False` no ejecuta evaluaciones ni entrenamiento.
El baseline completo requiere recorrer train; su coste es E/S, no entrenamiento.


In [ ]:
# @title Selección acotada
DEVICE = "cuda" # @param ["cuda", "cpu"]
RUN_SHORT_PROBE = False # @param {type:"boolean"}
EXECUTE = False # @param {type:"boolean"}
run_id = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
OUTPUT_DIR = Path(OUTPUT_PARENT) / f"diagnostic_{run_id}"
command = [sys.executable, "-m", "src.latent_diagnostic_runner",
           "--processed-dir", str(PROCESSED_DIR), "--output-dir", str(OUTPUT_DIR),
           "--settings", str(WORKSHOP_DIR / "config/latent_information_phase3.json"),
           "--device", DEVICE]
for label, checkpoint in (("best", BEST_CHECKPOINT), ("last", LAST_CHECKPOINT)):
    if checkpoint.strip():
        command += ["--checkpoint", label + "=" + str(Path(checkpoint).expanduser().resolve())]
if RUN_SHORT_PROBE:
    command.append("--short-probe")
print("Salida nueva:", OUTPUT_DIR)
subprocess.run(command, cwd=WORKSHOP_DIR, check=True, timeout=30)


In [ ]:
if EXECUTE:
    completed = subprocess.run(command + ["--execute"], cwd=WORKSHOP_DIR,
                               check=False, timeout=650)
    print("Código de salida:", completed.returncode)
    summary_path = OUTPUT_DIR / "summary.json"
    if summary_path.exists():
        summary = json.loads(summary_path.read_text())
        print("Estado:", summary["status"], summary.get("reason", ""))
    else:
        print("Sin resumen final: corte duro. Los archivos existentes son parciales.")
else:
    print("Solo plan. Activa EXECUTE cuando quieras ejecutar este diagnóstico.")


## Comparaciones pareadas y posiciones

La pérdida es CE contra **x**, suma de 80 posiciones y media de ejemplos:
nats/secuencia. Dividir por 80 da nats/carácter. UNK=0 participa; no hay PAD.
`swap − original > 0` indica que el código alineado ayudaba a reconstruir.
TV/JS detectan cambios de distribución aunque argmax no cambie. Los logits
centrados eliminan su constante aditiva por posición. `z=0` puede quedar fuera
de la distribución; se interpreta como intervención secundaria.

La sensibilidad antigua se conserva en el primer batch de 32 y sobre todo el
subconjunto. Su escala no mide información mutua ni utilidad reconstructiva.
Los deltas y su dispersión son descriptivos: ventanas solapadas/prefijo fijo
no justifican intervalos de confianza IID. Ver también acuerdo donante/receptor.


In [ ]:
import pandas as pd
reports = []
if EXECUTE and (OUTPUT_DIR / "summary.json").exists():
    for p in sorted(OUTPUT_DIR.glob("checkpoint_*.json")):
        record = json.loads(p.read_text())
        reports.append((record["label"], None, record["diagnostics"]))
    for p in sorted(OUTPUT_DIR.glob("short_*/step_*.json")):
        reports.append((p.parent.name, int(p.stem.split("_")[1]), json.loads(p.read_text())))
    rows = []
    for label, step, d in reports:
        c = d["comparisons"]["swapped_minus_original"]
        rows.append({"modelo": label, "update": step,
                     "AE nats/seq": d["modes"]["original"]["nats_per_sequence"],
                     "marginal nats/seq": d["baseline"]["marginal"]["nats_per_sequence"],
                     "posición nats/seq": d["baseline"]["position_marginal"]["nats_per_sequence"],
                     "swap-original nats/seq": c["delta_nats_per_sequence"]["mean"],
                     "TV swap": c["probability_tv_mean"],
                     "top1 cambia": c["top1_change_fraction"],
                     "sensibilidad antigua 32": d["legacy_first_batch"]["latent_sensitivity"],
                     "z std": d["latent"]["between_example_std_rms"],
                     "acuerdo tokens swap": d["swapped_source_target_token_agreement"]})
    display(pd.DataFrame(rows))
    baseline_path = OUTPUT_DIR / "validation_baseline.json"
    if not reports and baseline_path.exists():
        b = json.loads(baseline_path.read_text())
        display(pd.DataFrame([{ "baseline": k, "nats/secuencia": b[k]["nats_per_sequence"],
                                "nats/carácter": b[k]["nats_per_character"]}
                              for k in ("marginal", "position_marginal")]))


In [ ]:
import matplotlib.pyplot as plt
if reports:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))
    for label, step, d in reports:
        tag = f"{label} / {step}" if step is not None else label
        position = range(1, d["sequence_length"]+1)
        c = d["comparisons"]["swapped_minus_original"]["per_position"]
        axes[0].plot(position, c["delta_nats_per_character"], label=tag)
        axes[1].plot(position, c["probability_tv"], label=tag)
        axes[2].plot(position, d["modes"]["original"]["per_position_nats_per_character"], label=tag)
    last = reports[-1][2]
    axes[2].plot(position, last["baseline"]["marginal"]["per_position_nats_per_character"], "k--", label="marginal")
    axes[2].plot(position, last["baseline"]["position_marginal"]["per_position_nats_per_character"], "k:", label="posición")
    for ax, title in zip(axes, ("Daño por swap (nats/carácter)", "TV de probabilidades", "Reconstrucción (nats/carácter)")):
        ax.set(xlabel="Posición (1–80)", title=title)
        ax.grid(alpha=.2)
    axes[0].axhline(0, color="black", lw=.7)
    axes[2].legend(fontsize=6, bbox_to_anchor=(1.02, 1))
    plt.tight_layout()
    plt.show()


## Interpretación y siguiente decisión

- Pérdida próxima al baseline, TV pequeña frente a él, escasa diversidad entre
  ejemplos y daño por swap próximo a cero: evidencia de predictor de frecuencias
  sobre este subconjunto. Examina también los 80 puntos y los tercios.
- Un daño por swap positivo, relevante y reproducible apoya el uso de información
  útil: revisar dispersión y controles, aunque cambie poco el argmax o la métrica
  antigua sea pequeña. Un promedio positivo diminuto no basta. TV alta sin daño
  relevante es dependencia sin utilidad reconstructiva demostrada.
- Mejor que la marginal global pero próxima a la marginal por posición puede
  indicar aprendizaje de posición, sin dependencia informativa de `z`.

Tras completar las evaluaciones, consulta `LATENT_INFORMATION_REPORT.md` para
las alternativas condicionadas. No se selecciona arquitectura ni se inicia AE
z64/VAE automáticamente. Para revisar resultados usa JSON; para compartirlos
descarga/copias la carpeta nueva de Drive. El estado `stopped/failed/partial`
requiere revisar el motivo; no equivale a un experimento completado.
